# TP : Agentic AI avec LangChain et Gemini

Version adaptée du notebook `AI_Agentic.ipynb` pour utiliser **Gemini via l’API Google AI Studio**. Le modèle n’est pas exécuté localement : les requêtes sont envoyées à Google et nécessitent Internet ainsi qu’une clé API.


## 0. Imports et configuration de Gemini

La cellule charge `GOOGLE_API_KEY` depuis le fichier `.env`. Si elle ne la trouve pas, elle demande la clé sans l’afficher à l’écran. Le modèle utilisé par défaut est `gemini-3.6-flash`, le modèle testé avec succès dans ton environnement ; si ce modèle n’est pas disponible pour ton projet, vérifie les modèles accessibles dans [la documentation officielle](https://ai.google.dev/gemini-api/docs/models).


In [8]:
import os
from getpass import getpass
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Charger les variables présentes dans le fichier .env du dossier courant.
load_dotenv(override=False)

# Pour utiliser une clé Google AI Studio (Gemini Developer API), et non Vertex AI.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "false"

# Compatibilité : les intégrations Google acceptent GOOGLE_API_KEY ou GEMINI_API_KEY.
if not os.getenv("GOOGLE_API_KEY") and os.getenv("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

# Si aucune clé n'est configurée, demander la clé en saisie masquée.
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Colle ta clé API Gemini (saisie masquée) : ").strip()

def text_content(content):
    """Convertit le contenu texte Gemini en chaîne lisible."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        chunks = []
        for block in content:
            if isinstance(block, str):
                chunks.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                chunks.append(block.get("text", ""))
        return "\n".join(part for part in chunks if part) or str(content)
    return str(content)

MODEL_NAME = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    # Gemini 3.x compte aussi les tokens de réflexion dans max_output_tokens.
    # Une réflexion basse et un plafond plus élevé évitent les réponses coupées.
    thinking_level="low",
    max_output_tokens=4096,
    timeout=60,
    max_retries=2,
)

# Test de connexion : une réponse doit s'afficher si la clé, le modèle et l'accès API sont valides.
print(text_content(llm.invoke("Réponds en une phrase : qu'est-ce qu'un agent IA ?").content))


Un agent IA est un programme informatique autonome capable d'analyser son environnement, de prendre des décisions et de réaliser des actions pour atteindre un objectif précis.


## 1. Agent simple

Un agent combine un modèle, un prompt système et, éventuellement, des outils. Ici, aucun outil n’est nécessaire.

In [9]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds clairement en français."
)

response = agent.invoke({
    "messages": [HumanMessage(content="Présente-toi en deux phrases.")]
})
print(text_content(response["messages"][-1].content))

Je suis un assistant virtuel conçu pour vous aider, répondre à vos questions et vous accompagner au quotidien. Mon objectif est de vous fournir des informations claires, précises et utiles à tout moment.


## 2. Configuration du modèle

Avec `gemini-3.6-flash`, `thinking_level="low"` réduit le temps consacré à la réflexion pour les questions simples. `max_output_tokens` est un plafond qui comprend les tokens de réflexion ET ceux de la réponse finale. Une valeur trop petite peut couper la réponse avant qu'elle soit terminée. Les exemples suivants utilisent donc un plafond plus élevé.


In [10]:
llm_precis = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    thinking_level="low",
    max_output_tokens=4096,
    timeout=60,
    max_retries=2,
)
print(text_content(llm_precis.invoke("Explique ce qu'est Python en trois points.").content))


Voici ce qu'est Python en trois points simples :

1. **Un langage simple et facile à apprendre :** Python est un langage de programmation connu pour sa syntaxe claire et lisible, très proche de l'anglais. Il permet d'écrire du code rapidement avec moins de lignes que la plupart des autres langages, ce qui en fait le choix idéal pour les débutants.

2. **Extrêmement polyvalent :** C'est un langage "tout-terrain". On l'utilise aussi bien pour créer des sites web et automatiser des tâches répétitives, que pour des domaines avancés comme la science des données (Data Science), la finance ou l'intelligence artificielle (ex: ChatGPT a été développé principalement avec Python).

3. **Gratuit et soutenu par une immense communauté :** Python est un logiciel libre (*open-source*). Il dispose d'un écosystème gigantesque de "bibliothèques" (des modules de code prêts à l'emploi créés par la communauté) qui permettent de réaliser presque n'importe quel projet sans avoir à tout réinventer.


## 3. Sélection dynamique du modèle

Le middleware `@wrap_model_call` choisit une configuration au moment de l’appel, selon le champ `context`. Pour éviter de dépendre de plusieurs modèles qui pourraient avoir des accès ou quotas différents, les deux configurations utilisent ici `gemini-3.6-flash` mais avec des températures différentes. Tu peux remplacer l’un des noms de modèle par un autre modèle auquel ton projet a accès.


In [11]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# Deux configurations du même modèle.
# Le plafond inclut les tokens de réflexion : on évite les très petites valeurs.
basic_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME, thinking_level="low", max_output_tokens=4096,
    timeout=60, max_retries=2,
)
advanced_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME, thinking_level="low", max_output_tokens=8192,
    timeout=60, max_retries=2,
)

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    env = request.runtime.context.get("env", "test")
    selected_model = advanced_model if env == "prod" else basic_model
    selected_limit = 8192 if env == "prod" else 4096
    print(
        f"Contexte={env}; modèle={MODEL_NAME}; "
        f"plafond={selected_limit} tokens; réflexion=low"
    )
    return handler(request.override(model=selected_model))

agent_dynamic = create_agent(
    model=basic_model,
    tools=[],
    middleware=[dynamic_model_selection],
)

r_test = agent_dynamic.invoke(
    {"messages": [HumanMessage(content="Explique brièvement les réseaux neuronaux.")]},
    context={"env": "test"},
)
print(text_content(r_test["messages"][-1].content))


Contexte=test; modèle=gemini-3.6-flash; plafond=4096 tokens; réflexion=low
Un **réseau neuronal** (ou réseau de neurones artificiels) est un système informatique inspiré du fonctionnement du **cerveau humain**. C'est la technologie clé derrière l'intelligence artificielle moderne et le *Deep Learning*.

Voici son fonctionnement résumé en 3 points :

1. **La Structure (Les couches) :** 
   Le réseau est composé de "neurones" virtuels connectés entre eux et organisés en couches :
   * **Couche d'entrée :** reçoit la donnée brute (ex: les pixels d'une photo).
   * **Couches cachées :** analysent l'information étape par étape (ex: détection des formes, des couleurs, puis des visages).
   * **Couche de sortie :** donne le résultat (ex: "C'est un chat").

2. **L'Apprentissage :** 
   Le réseau apprend par l'exemple. On lui montre des milliers d'images. Au début, il se trompe, mais il **ajuste la force de ses connexions** (appelées "poids") à chaque erreur pour devenir de plus en plus précis.

In [12]:
r_prod = agent_dynamic.invoke(
    {"messages": [HumanMessage(content="Explique brièvement les réseaux neuronaux.")]},
    context={"env": "prod"},
)
print(text_content(r_prod["messages"][-1].content))

Contexte=prod; modèle=gemini-3.6-flash; plafond=8192 tokens; réflexion=low
Un **réseau de neurones artificiels** est un système informatique inspiré du fonctionnement du cerveau humain. C'est la technologie clé derrière l'intelligence artificielle moderne (l'apprentissage profond ou *Deep Learning*).

Voici comment cela fonctionne en résumé :

1. **La structure** : Il est composé de "neurones" virtuels organisés en **couches** :
   * **Couche d'entrée** : elle reçoit les données brutes (ex. : une photo).
   * **Couches cachées** : elles analysent les détails (ex. : formes, couleurs, contours) via des calculs mathématiques.
   * **Couche de sortie** : elle donne la réponse finale (ex. : "C'est un chat").

2. **L'apprentissage** : Au début, le réseau fait des erreurs. On lui donne des milliers d'exemples et, à chaque erreur, il ajuste la force des connexions entre ses neurones (appelées les "poids"). Avec la pratique, il devient de plus en plus précis.

**À quoi ça sert ?** 
À reconnaîtr

## 4. Mémoire de conversation

`InMemorySaver` conserve l’état dans la mémoire du processus. Le même `thread_id` doit être réutilisé pour retrouver la conversation. Ce stockage est temporaire : il disparaît à l’arrêt du processus.

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
agent_memory = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile."
    ,checkpointer=memory,
)
config = {"configurable": {"thread_id": "conversation-1"}}

r1 = agent_memory.invoke(
    {"messages": [HumanMessage(content="Je m'appelle Mouad.")]},
    config=config,
)
print(text_content(r1["messages"][-1].content))

r2 = agent_memory.invoke(
    {"messages": [HumanMessage(content="Comment je m'appelle ?")]},
    config=config,
)
print(text_content(r2["messages"][-1].content))

Enchanté, Mouad ! Comment puis-je t'aider aujourd'hui ?
Tu t'appelles Mouad !


## 5. Outils (*tools*)

Un outil est une fonction Python décorée par `@tool`. Le modèle décide quand l’appeler à partir de sa description et des arguments. Les outils météo et recherche ci-dessous sont **des exemples fictifs** : ils ne consultent aucune source en temps réel.

In [15]:
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Recherche dans un petit ensemble de résultats d'exemple."""
    return f"Résultats d'exemple pour '{query}' : documentation, tutoriel, article de synthèse."

@tool
def get_weather(location: str) -> str:
    """Retourne une météo fictive pour démontrer l'appel d'un outil."""
    return f"Météo fictive pour {location} : ciel dégagé, 22 °C. Ceci n'est pas une donnée en temps réel."

@tool
def get_employee_info(name: str) -> str:
    """Retourne un profil fictif pour un exemple pédagogique."""
    return f"Profil fictif de {name} : poste Développeur, domaine Informatique."

agent_tools = create_agent(
    model=llm,
    tools=[search, get_weather, get_employee_info],
    system_prompt="Tu es un assistant. Utilise les outils lorsqu'ils sont utiles et indique quand une donnée est fictive."
)

r = agent_tools.invoke({
    "messages": [HumanMessage(content="Utilise l'outil météo pour donner l'exemple de météo pour Béni Mellal.")]
})
print(text_content(r["messages"][-1].content))

Voici un exemple de météo fictive pour Béni Mellal :

* **Conditions :** Ciel dégagé
* **Température :** 22 °C

*(Remarque : Il s'agit d'une donnée fictive générée pour la démonstration et non d'une information météo en temps réel.)*


In [20]:
# ============================================================
# CHANGEMENT GLOBAL DU MODÈLE GEMINI
# À exécuter juste avant la section 7. Outils
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

# Modèle principal utilisé par les agents
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.1,
    max_output_tokens=4096,
    timeout=30,
    max_retries=0,
    vertexai=False,
)

# Mettre également à jour les configurations du modèle dynamique
basic_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.1,
    max_output_tokens=1024,
    timeout=30,
    max_retries=0,
    vertexai=False,
)

advanced_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.1,
    max_output_tokens=2048,
    timeout=30,
    max_retries=0,
    vertexai=False,
)

print("Modèle configuré :", MODEL_NAME)
print("Agent principal et configurations dynamiques mis à jour.")

Modèle configuré : gemini-3.5-flash-lite
Agent principal et configurations dynamiques mis à jour.


## 6. Recherche Web facultative avec DuckDuckGo

Cette partie nécessite une connexion Internet et peut être limitée par le service. Elle n’exige pas de clé Tavily. Si la recherche échoue, continue le TP avec l’outil `search` d’exemple ci-dessus.

In [45]:
from ddgs import DDGS

@tool
def web_search(query: str, num_results: int = 5) -> str:
    """Recherche des pages Web via DuckDuckGo. Nécessite Internet."""
    try:
        results = DDGS().text(query, max_results=num_results)
        if not results:
            return f"Aucun résultat trouvé pour : {query}"
        lines = []
        for i, item in enumerate(results, start=1):
            title = item.get("title", "Sans titre")
            body = item.get("body", "Sans description")
            href = item.get("href", "")
            lines.append(f"{i}. {title} — {body} — {href}")
        return "\n".join(lines)
    except Exception as exc:
        return f"La recherche Web a échoué ({type(exc).__name__}). Vérifie la connexion puis réessaie."

web_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="Réponds en français. Pour les informations actuelles, utilise l'outil de recherche et distingue les résultats trouvés de tes explications."
)
# Exemple : décommente pour tester la recherche en ligne.
# result_web = web_agent.invoke({"messages": [HumanMessage(content="Recherche des tutoriels récents sur Agentic RAG.")]})
# print(result_web["messages"][-1].content)

In [46]:
result_web = web_agent.invoke({
    "messages": [
        HumanMessage(
            content="Recherche des tutoriels sur Agentic RAG."
        )
    ]
})

print(text_content(result_web["messages"][-1].content))

Bien que la recherche en direct ait rencontré un problème technique momentané, je peux vous orienter vers les meilleurs tutoriels, concepts et frameworks de référence pour maîtriser l'**Agentic RAG** (Retrieval-Augmented Generation piloté par des agents).

L'Agentic RAG va au-delà du RAG traditionnel (linéaire : question -> recherche -> génération) en donnant à un LLM des capacités d'agent (planification, utilisation d'outils, auto-correction, boucles de rétroaction).

Voici où trouver les meilleurs tutoriels et comment aborder le sujet :

---

### 1. Les Tutoriels de Référence par Framework

Les deux écosystèmes majeurs pour le développement d'agents et de RAG proposent d'excellents tutoriels gratuits et documentés :

*   **LlamaIndex (Guides "Agentic RAG")**
    *   *Pourquoi y aller :* LlamaIndex est pionnier dans la gestion des données pour LLM. Leurs guides montrent comment créer des agents capables de choisir dynamiquement entre plusieurs sources de données, d'effectuer de la rec

## 7. Outil Python REPL

Le REPL exécute du code Python dans le processus local. **Ne lui confie pas de code non fiable** et n’utilise pas ce mécanisme sur une machine contenant des données sensibles. Commence par des calculs simples.

In [22]:
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import Tool

python_repl = PythonREPL()
repl_tool = Tool(
    name="python_repl",
    description="Exécute du code Python simple pour des calculs. N'utiliser que pour des opérations de calcul sûres.",
    func=python_repl.run,
)

print(repl_tool.invoke("a = 5\nb = 9\nprint(a + b)"))

14



In [43]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage

MODEL_NAME = "gemini-3.5-flash-lite"

# Modèle configuré pour les appels d'outils
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    max_output_tokens=1024,
    timeout=120,
    max_retries=2,
    thinking_level="minimal",
    vertexai=False,
)

print("Modèle :", MODEL_NAME)
print("Délai d'attente : 120 secondes")

# Recréer l'agent avec le nouveau modèle
repl_agent = create_agent(
    model=llm,
    tools=[repl_tool],
    system_prompt=(
        "Pour les calculs simples, utilise l'outil Python. "
        "N'effectue aucune opération sur le système ou les fichiers. "
        "Après le calcul, donne une réponse courte."
    ),
)

print("Agent Python REPL recréé.")


Modèle : gemini-3.5-flash-lite
Délai d'attente : 120 secondes
Agent Python REPL recréé.


In [44]:
repl_agent = create_agent(
    model=llm,
    tools=[repl_tool],
    system_prompt="Pour les petits calculs, écris du code Python simple et utilise l'outil Python. Ne tente aucune opération sur le système ou les fichiers.",
)
r_repl = repl_agent.invoke({
    "messages": [HumanMessage(content="Utilise Python pour calculer la somme de 5 et 9.")]
})
print(text_content(r_repl["messages"][-1].content))

La somme de 5 et 9 est 14.


## 8. Gestion des erreurs des outils

`wrap_tool_call` intercepte les erreurs émises par un outil et retourne un message compréhensible à l’agent au lieu de laisser remonter immédiatement l’exception.

In [24]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@tool
def divide(a: float, b: float) -> str:
    """Divise a par b. Une division par zéro provoque une erreur."""
    if b == 0:
        raise ValueError("Le diviseur ne peut pas être zéro.")
    return str(a / b)

@wrap_tool_call
def handle_tool_errors(request, handler):
    try:
        return handler(request)
    except Exception as exc:
        return ToolMessage(
            content=f"Erreur de l'outil ({type(exc).__name__}). Vérifie les arguments.",
            tool_call_id=request.tool_call["id"],
        )

agent_errors = create_agent(
    model=llm,
    tools=[divide],
    middleware=[handle_tool_errors],
    system_prompt="Pour les divisions, utilise l'outil divide. Si l'outil renvoie une erreur, explique-la simplement."
)
r = agent_errors.invoke({"messages": [HumanMessage(content="Calcule 10 divisé par 0 avec l'outil.")]})
print(text_content(r["messages"][-1].content))

c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


La division par zéro est impossible (division par zéro). C'est pourquoi l'outil a renvoyé une erreur.


## 9. System prompt dynamique selon le rôle

Le contexte est transmis à l’agent au moment de l’appel et utilisé pour construire le prompt système.

In [25]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    role = request.runtime.context.get("user_role", "user")
    if role == "expert":
        return "Tu es un assistant technique. Donne des détails et utilise la terminologie du domaine. Réponds en français."
    if role == "beginner":
        return "Tu es un assistant pédagogique. Explique simplement, définis les termes et donne un exemple. Réponds en français."
    return "Tu es un assistant utile. Réponds clairement en français."

role_agent = create_agent(
    model=llm,
    tools=[],
    middleware=[user_role_prompt],
    context_schema=Context,
)

r_beginner = role_agent.invoke(
    {"messages": [HumanMessage(content="Explique un agent IA.")]},
    context={"user_role": "beginner"},
)
print(text_content(r_beginner["messages"][-1].content))

c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Bonjour ! C'est une excellente question. 

Pour comprendre ce qu'est un **agent IA (Intelligence Artificielle)**, imagine un petit robot virtuel ou un assistant intelligent. 

### 🌟 Explication simple
Un agent IA est un programme informatique conçu pour **observer** son environnement, **réfléchir** à la situation et **agir** de manière autonome pour atteindre un objectif précis. 

Contrairement à un logiciel classique qui fait <u>toujours</u> exactement la même chose (comme une calculatrice), un agent IA peut s'adapter, apprendre de ses erreurs et prendre des décisions tout seul.

---

### 📖 Définition des termes clés
Pour bien saisir le concept, voici comment fonctionne un agent IA (le modèle "Perception-Raisonnement-Action") :
1. **L'environnement :** C'est le monde (réel ou virtuel) dans lequel évolue l'agent.
2. **La perception :** C'est la façon dont l'agent "voit" ou "entend" son environnement (grâce à des capteurs, des micros, ou en lisant des données).
3. **Le raisonnement :** 

In [42]:
r_expert = role_agent.invoke(
    {"messages": [HumanMessage(content="Explique un agent IA.")]},
    context={"user_role": "expert"},
)
print(text_content(r_expert["messages"][-1].content))

c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


En intelligence artificielle, un **agent IA** est une entité autonome conçue pour percevoir son environnement au moyen de capteurs, traiter ces informations, prendre des décisions et agir sur cet environnement à l'aide d'actionneurs afin d'atteindre un objectif spécifique ou d'optimiser une fonction de performance.

Voici une décomposition détaillée de sa structure, de son fonctionnement et de sa taxonomie selon les standards de l'ingénierie des systèmes autonomes.

---

### 1. Architecture fondamentale d'un Agent IA

L'architecture d'un agent peut se résumer par la fonction mathématique suivante : à partir d'une séquence de perceptions (l'historique de ce que l'agent a capté depuis le début), l'agent associe une action. 

$$\text{Action} : P^* \rightarrow A$$
*(où $P^*$ est l'espace des séquences de perceptions et $A$ l'espace des actions).*

Dans le détail, un agent est généralement composé de quatre modules principaux :

*   **Les Capteurs (Perception) :** Ils permettent à l'agent d

## 10. Sortie structurée

Au lieu de demander seulement du texte libre, définis un schéma Pydantic. LangChain valide alors la structure retournée. Les sorties structurées dépendent des capacités d’appel d’outils du modèle Gemini sélectionné.

In [27]:
from pydantic import BaseModel
from langchain.agents.structured_output import ToolStrategy

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

contact_agent = create_agent(
    model=llm,
    tools=[],
    response_format=ToolStrategy(ContactInfo),
    system_prompt="Extrais fidèlement les champs demandés. N'invente pas les données manquantes."
)
contact_result = contact_agent.invoke({
    "messages": [HumanMessage(content="Extrais les champs de cet exemple : Mouad Aguelloul, mouad@example.com, +212 600 000 000.")]
})
contact = contact_result["structured_response"]
print(contact.name)
print(contact.email)
print(contact.phone)

c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Mouad Aguelloul
mouad@example.com
+212 600 000 000


## 11. Middleware personnalisé et hooks

Les hooks ci-dessous illustrent le schéma fourni : `before_agent` s’exécute au début d’une invocation ; `before_model` avant un appel au modèle ; `after_model` après la réponse du modèle ; `after_agent` à la fin du traitement. Les hooks de modèle peuvent se répéter si l’agent effectue plusieurs cycles outil/modèle.

In [28]:
import time
from langchain.agents.middleware import AgentMiddleware, AgentState

class HooksDemo(AgentMiddleware):
    def __init__(self):
        super().__init__()
        self.start_time = 0.0

    def before_agent(self, state: AgentState, runtime) -> None:
        self.start_time = time.perf_counter()
        print("[hook] before_agent")

    def before_model(self, state: AgentState, runtime) -> None:
        print("[hook] before_model")

    def after_model(self, state: AgentState, runtime) -> None:
        print("[hook] after_model")

    def after_agent(self, state: AgentState, runtime) -> None:
        elapsed = time.perf_counter() - self.start_time
        print(f"[hook] after_agent — durée approx. {elapsed:.2f} s")

hooks_agent = create_agent(model=llm, tools=[], middleware=[HooksDemo()])
r_hooks = hooks_agent.invoke({"messages": [HumanMessage(content="Combien font 12 + 30 ?")]})
print(text_content(r_hooks["messages"][-1].content))

[hook] before_agent
[hook] before_model


c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[hook] after_model
[hook] after_agent — durée approx. 2.19 s
12 + 30 font **42**.


## 12. RAG comme outil de l’agent

Le RAG combine la recherche dans des documents avec la génération par le modèle. On utilise ici `gemini-embedding-001` via l’API Gemini pour transformer les textes en vecteurs et FAISS pour retrouver les passages les plus proches. Cette étape utilise aussi la clé API et peut consommer du quota.


In [29]:
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool

texts = [
    "Le projet Alpha est un assistant local basé sur Gemini.",
    "Les données du projet sont stockées dans une base documentaire de démonstration.",
    "Le RAG recherche des passages pertinents avant de générer une réponse.",
    "LangChain permet de connecter un modèle à des outils Python.",
    "FAISS indexe les vecteurs pour accélérer la recherche de similarité.",
]
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = FAISS.from_texts(texts=texts, embedding=embedding_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Vérifier d'abord la recherche sans agent.
for doc in retriever.invoke("Que signifie RAG ?"):
    print("-", doc.page_content)

C:\Users\mouad\AppData\Local\Temp\ipykernel_1560\1269673553.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


- Le RAG recherche des passages pertinents avant de générer une réponse.
- Les données du projet sont stockées dans une base documentaire de démonstration.
- LangChain permet de connecter un modèle à des outils Python.


In [30]:
retrieval_tool = create_retriever_tool(
    retriever,
    name="kb_search",
    description="Recherche dans la base documentaire locale du TP. Utilise cet outil pour répondre aux questions sur le projet.",
)
rag_agent = create_agent(
    model=llm,
    tools=[retrieval_tool],
    system_prompt="Réponds à partir des documents retrouvés. Si l'information n'est pas dans la base, dis-le explicitement."
)
rag_result = rag_agent.invoke({
    "messages": [HumanMessage(content="Que signifie RAG et quelle bibliothèque indexe les vecteurs dans cette base ?")]
})
print(text_content(rag_result["messages"][-1].content))

c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\mouad\OneDrive\Desktop\AIDC\2emeannee\S3\DSD\Tps\AI_Agentic_Gemini\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**RAG** signifie **Retrieval-Augmented Generation** (génération augmentée par récupération). Il s'agit d'une technique qui consiste à rechercher des passages pertinents dans une base documentaire avant de les transmettre à un modèle pour générer une réponse précise.

La bibliothèque utilisée pour indexer les vecteurs et accélérer la recherche de similarité dans cette base est **FAISS**.


## 13. Garde-fous : détection de données personnelles (PII)

Le middleware intégré peut masquer ou bloquer des informations personnelles avant leur transmission au modèle. Utilise des données fictives pour les tests. Les outils ci-dessous ne réalisent ni paiement ni envoi d’e-mail.

In [33]:
from langchain.agents.middleware import PIIMiddleware

pii_agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
    system_prompt="Réponds brièvement en français. Ne tente pas de deviner les informations masquées."
)
pii_result = pii_agent.invoke({
    "messages": [HumanMessage(content="Mon e-mail de test est test@example.com et ma carte fictive est 4111-1111-1111-1111. Explique ce qui s'est passé.")]
})
print(text_content(pii_result["messages"][-1].content))

Je ne peux pas voir votre e-mail ni les chiffres de votre carte car ces informations ont été masquées pour des raisons de sécurité. 

Il semble que vous ayez effectué un test en utilisant des données fictives, mais en l'absence de détails sur le contexte ou le système utilisé, je ne peux pas expliquer ce qui s'est passé. Pouvez-vous préciser de quel outil ou application il s'agit ?


## 14. Human-in-the-loop : validation avant l’outil

L’agent s’interrompt avant l’exécution de l’outil sensible. La reprise doit réutiliser le même `thread_id`. L’outil d’e-mail de cet exemple est simulé et n’envoie aucun message réel.

In [36]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def send_email(email_address: str) -> str:
    """Simule l'envoi d'un e-mail ; n'envoie aucun e-mail réel."""
    print(f"[SIMULATION] Envoi d'e-mail à {email_address}")
    return f"Simulation terminée pour {email_address}. Aucun e-mail réel n'a été envoyé."

approval_agent = create_agent(
    model=llm,
    tools=[send_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
    checkpointer=InMemorySaver(),
    system_prompt="Si l'utilisateur demande un e-mail, prépare l'appel de l'outil send_email. L'outil est une simulation pédagogique."
)
approval_config = {"configurable": {"thread_id": "approval-demo-1"}}
approval_result = approval_agent.invoke(
    {"messages": [HumanMessage(content="Prépare l'envoi d'un e-mail à mouadag10@gmail.com avec l'outil send_email.")]},
    config=approval_config,
)
print(approval_result.get("__interrupt__", "Aucune interruption affichée : vérifie que le modèle a appelé l'outil."))

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'email_address': 'mouadag10@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'email_address': 'mouadag10@gmail.com'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='2246cf67a1b5870929596436a20d50f2', response_schema=None)]


In [37]:
# Ne lance cette cellule qu'après avoir vérifié le contenu de l'interruption ci-dessus.
# Elle APPROUVE l'appel en attente et exécute l'outil simulé.
# Si aucune interruption n'a eu lieu, ne lance pas la reprise.
if approval_result.get("__interrupt__"):
    approval_result = approval_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=approval_config,
    )
    print(text_content(approval_result["messages"][-1].content))
else:
    print("Pas d'appel en attente : aucune reprise nécessaire.")

[SIMULATION] Envoi d'e-mail à mouadag10@gmail.com
L'appel à l'outil `send_email` a été effectué pour l'adresse **mouadag10@gmail.com** (simulation pédagogique, aucun e-mail réel n'a été envoyé).


## 15. Garde-fou personnalisé avant l’agent

Ce hook filtre certains mots-clés avant tout appel au modèle. C’est une démonstration déterministe, pas un système général de modération : la liste et la logique doivent être adaptées au cas d’usage.

In [38]:
from typing import Any
from langchain.agents.middleware import before_agent, AgentState
from langgraph.runtime import Runtime

banned_keywords = ["mot-interdit-demo", "requete-bloquee-demo"]

@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if not state["messages"]:
        return None
    first_user_message = next((m for m in state["messages"] if getattr(m, "type", None) == "human"), None)
    if first_user_message is None or not isinstance(first_user_message.content, str):
        return None
    content = first_user_message.content.lower()
    if any(word in content for word in banned_keywords):
        return {
            "messages": [{"role": "assistant", "content": "Cette requête a été bloquée par le filtre de démonstration."}],
            "jump_to": "end",
        }
    return None

filter_agent = create_agent(model=llm, tools=[], middleware=[content_filter])
blocked_result = filter_agent.invoke({"messages": [HumanMessage(content="Test avec mot-interdit-demo.")]})
print(text_content(blocked_result["messages"][-1].content))

Cette requête a été bloquée par le filtre de démonstration.


## 16. Garde-fou après l’agent

Un hook après l’agent peut examiner la réponse finale avant sa restitution. Ici, un second appel à Gemini joue le rôle de juge et renvoie seulement `SAFE` ou `UNSAFE`. Ce contrôle probabiliste ne remplace pas une validation de sécurité formelle et consomme une requête API supplémentaire.


In [39]:
from langchain.agents.middleware import after_agent

safety_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    thinking_level="minimal",
    max_output_tokens=512,
    timeout=60,
    max_retries=2,
)

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if not state["messages"]:
        return None
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None
    prompt = (
        "Évalue si la réponse suivante est sûre et appropriée. "
        "Réponds uniquement SAFE ou UNSAFE.\nRéponse : " + text_content(last_message.content)
    )
    judgement = text_content(safety_model.invoke([HumanMessage(content=prompt)]).content).strip().upper()
    if "UNSAFE" in judgement:
        return {"messages": [AIMessage(content="Je ne peux pas fournir cette réponse. Reformule ta demande.")]}
    return None

safety_agent = create_agent(model=llm, tools=[], middleware=[safety_guardrail])
safe_result = safety_agent.invoke({"messages": [HumanMessage(content="Explique à quoi sert un pare-feu informatique.")]})
print(text_content(safe_result["messages"][-1].content))


Un **pare-feu** (ou *firewall* en anglais) est un système de sécurité informatique qui sert de **barrière de protection** entre votre ordinateur (ou votre réseau local) et l'extérieur (généralement Internet). 

Pour faire simple, il agit comme un **gardien de sécurité** ou un agent de douane à l'entrée de votre réseau. Son rôle principal est de filtrer le trafic de données pour empêcher les intrus d'entrer et d'empêcher les données sensibles de sortir sans autorisation.

Voici en détail à quoi il sert :

### 1. Contrôler le trafic entrant et sortant
Le pare-feu analyse toutes les données (sous forme de paquets) qui tentent d'entrer ou de sortir de votre appareil. Pour ce faire, il s'appuie sur des **règles de sécurité** prédéfinies :
* **Ce qui est autorisé :** Par exemple, il laisse passer le trafic de votre navigateur web (quand vous allez sur Google ou YouTube) parce que c'est vous qui l'avez demandé.
* **Ce qui est bloqué :** Il bloque les connexions suspectes ou non sollicitées pr